# C-max20-ES — exploratory Dev-only optimization

Run this only after the controlled factorial artifacts and conclusion have been frozen. This is a fresh Model-A-to-expanded-pool trajectory with maximum 20 epochs, minimum 8 epochs, Dev-loss patience 4, and min_delta 1e-4. It is not a factorial cell and never loads Test.

In [ ]:
from pathlib import Path
import json, shutil, subprocess
REPO = Path('/kaggle/working/VisolexNorm')
SOURCE_REF = 'main'  # replace with the same committed controlled-experiment revision
URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, URL, str(REPO)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
%cd {REPO}
!pip install -q -r requirements-kaggle.txt
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('Frozen source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
MOUNT = Path('/kaggle/input/visolexnorm-controlled-input')  # same Dev-only private input Dataset
WORK = Path('/kaggle/working/c_max20_es')
ARCHIVE = MOUNT/'controlled_training_input.zip'
DATA = WORK/'input' if ARCHIVE.is_file() else MOUNT
if ARCHIVE.is_file():
    if DATA.exists(): shutil.rmtree(DATA)
    shutil.unpack_archive(ARCHIVE, DATA)
required = [DATA/'checkpoints/model_a/config.json', DATA/'data/processed/vilexnorm_train.jsonl', DATA/'data/processed/vilexnorm_dev.jsonl', DATA/'data/processed/visolex_weak_labeled_expanded.jsonl']
assert not [str(path) for path in required if not path.is_file()]
assert not (DATA/'data/processed/vilexnorm_test.jsonl').exists()
assert not (DATA/'outputs/evaluation').exists()

In [ ]:
# Copy the frozen protocol downloaded from the completed factorial run to this Kaggle session before this cell.
PROTOCOL = Path('/kaggle/input/controlled-factorial-artifacts/controlled_factorial_protocol.json')  # update dataset/filename
assert PROTOCOL.is_file(), 'Attach the frozen factorial protocol artifact.'
RUN = WORK/'seed_2026'
!python -m scripts.controlled_experiments build-c-max20 --data-root {DATA} --config {REPO}/configs/c_max20_early_stopping_config.json --protocol {PROTOCOL} --seed 2026 --output {RUN}/mixture_manifest.json

In [ ]:
# Smoke test only; full run below starts freshly from Model A.
SMOKE = WORK/'smoke'
!python -m scripts.controlled_experiments train-c-max20 --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --manifest {RUN}/mixture_manifest.json --config {REPO}/configs/c_max20_early_stopping_config.json --work-dir {SMOKE} --smoke-test
smoke = json.loads((SMOKE/'smoke_test.json').read_text())
assert smoke['passed'] and smoke['test_inputs_loaded'] is False, smoke

In [ ]:
# If Kaggle interrupts, rerun this command with --resume and exactly the same RUN directory.
!python -m scripts.controlled_experiments train-c-max20 --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --manifest {RUN}/mixture_manifest.json --config {REPO}/configs/c_max20_early_stopping_config.json --work-dir {RUN}

In [ ]:
report = json.loads((RUN/'early_stopping_report.json').read_text())
assert report['rule']['min_epochs'] == 8 and report['rule']['patience'] == 4
assert report['test_inputs_loaded'] is False
shutil.rmtree(RUN/'state')  # archive does not need resumable optimizer state after a completed run
shutil.make_archive('/kaggle/working/c_max20_es_artifacts', 'zip', WORK, 'seed_2026')
print(report)
print('Download c_max20_es_artifacts.zip. This result is exploratory optimization, not a causal factorial result.')